<a href="https://colab.research.google.com/github/vneumannufprbr/TrabajosenPython/blob/main/Generador_de_Datos_Sint%C3%A9ticos_7_anios_Incidentes_PNP_ok_v1_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# -*- coding: utf-8 -*-
"""
Generador de Datos Sintéticos Completo

Objetivo: Tomar el dataset 'incidentes_simulados_restricciones.csv',
          analizar la estructura y distribuciones de TODAS sus columnas,
          y generar datos sintéticos para los 6 años anteriores,
          manteniendo la consistencia estadística y el formato exacto.
"""

# -----------------------------------------------------------------------------
# Paso 1: Importación de Librerías y Carga de Datos
# -----------------------------------------------------------------------------
import pandas as pd
import numpy as np

print("Librerías importadas correctamente.")

try:
    df_original = pd.read_csv('incidentes_simulados_restricciones.csv', encoding='utf-8')
    df_original['fecha'] = pd.to_datetime(df_original['fecha'])
    print(f"Cargados {len(df_original)} registros del archivo original.")
except FileNotFoundError:
    print("\nERROR: El archivo 'incidentes_simulados_restricciones.csv' no se encontró.")
    exit()

# -----------------------------------------------------------------------------
# Paso 2: Análisis Completo del Dataset Original
# -----------------------------------------------------------------------------
print("\nAnalizando el dataset original para aprender patrones y rangos...")

# --- Información General ---
start_year_original = df_original['fecha'].min().year
incidentes_por_anio = df_original.shape[0] // df_original['fecha'].dt.year.nunique()
print(f"Año de inicio de datos originales: {start_year_original}")
print(f"Se generarán ~{incidentes_por_anio} incidentes por cada nuevo año.")

# --- Aprender Rangos y Distribuciones de TODAS las columnas ---

# Coordenadas
lat_min, lat_max = df_original['lat'].min(), df_original['lat'].max()
lon_min, lon_max = df_original['lon'].min(), df_original['lon'].max()

# Columnas categóricas y sus probabilidades
prob_tipo_incidente = df_original['tipo_incidente'].value_counts(normalize=True)
prob_tipo_arma = df_original['tipo_arma_fuego'].value_counts(normalize=True)
prob_tipo_local = df_original['tipo_local'].value_counts(normalize=True)

# Datos de la Víctima
prob_sexo_victima = df_original['sexo_victima'].value_counts(normalize=True)
prob_antecedentes_victima = df_original['antecedentes_victima'].value_counts(normalize=True)
edad_victima_min, edad_victima_max = int(df_original['edad_victima'].min()), int(df_original['edad_victima'].max())

# Datos del Victimario (aprendiendo solo de los casos donde existen)
df_con_victimario = df_original.dropna(subset=['edad_victimario'])
prob_sexo_victimario = df_con_victimario['sexo_victimario'].value_counts(normalize=True)
prob_antecedentes_victimario = df_con_victimario['antecedentes_victimario'].value_counts(normalize=True)
prob_reiterante = df_con_victimario['reiterante'].value_counts(normalize=True)
edad_victimario_min, edad_victimario_max = int(df_con_victimario['edad_victimario'].min()), int(df_con_victimario['edad_victimario'].max())

# Probabilidad de aprehensión
prob_aprehension = df_original['aprehension_detencion_victimario'].value_counts(normalize=True)

print("Análisis de patrones y distribuciones completado.")

# -----------------------------------------------------------------------------
# Paso 3: Generación de Datos Sintéticos para 6 Años Anteriores
# -----------------------------------------------------------------------------
print("\nGenerando datos sintéticos para los 6 años anteriores...")

nuevos_incidentes = []
for i in range(1, 7):
    anio_actual = start_year_original - i
    print(f"  - Generando datos para el año {anio_actual}...")

    num_incidentes_anio = int(incidentes_por_anio * np.random.uniform(0.95, 1.05))

    start_date = pd.to_datetime(f'{anio_actual}-01-01')
    end_date = pd.to_datetime(f'{anio_actual}-12-31')
    random_dates = pd.to_datetime(np.random.randint(start_date.value, end_date.value, num_incidentes_anio))

    for fecha in random_dates:
        # Generar un incidente completo
        incidente = {
            'lat': np.random.uniform(lat_min, lat_max),
            'lon': np.random.uniform(lon_min, lon_max),
            'fecha': fecha,
            'hora': np.random.randint(0, 24),
            'minuto': np.random.randint(0, 60),
            'dia_semana': fecha.dayofweek,
            'tipo_incidente': np.random.choice(prob_tipo_incidente.index, p=prob_tipo_incidente.values),
            'tipo_arma_fuego': np.random.choice(prob_tipo_arma.index, p=prob_tipo_arma.values),
            'tipo_local': np.random.choice(prob_tipo_local.index, p=prob_tipo_local.values),
            'edad_victima': np.random.randint(edad_victima_min, edad_victima_max + 1),
            'sexo_victima': np.random.choice(prob_sexo_victima.index, p=prob_sexo_victima.values),
            'cip_victima': np.random.randint(1000000, 8000000), # CIP Ficticio
            'antecedentes_victima': np.random.choice(prob_antecedentes_victima.index, p=prob_antecedentes_victima.values),
            'aprehension_detencion_victimario': np.random.choice(prob_aprehension.index, p=prob_aprehension.values)
        }

        # Generar datos del victimario solo si hubo aprehensión
        if incidente['aprehension_detencion_victimario'] == 'Si':
            incidente['edad_victimario'] = np.random.randint(edad_victimario_min, edad_victimario_max + 1)
            incidente['sexo_victimario'] = np.random.choice(prob_sexo_victimario.index, p=prob_sexo_victimario.values)
            incidente['cip_victimario'] = np.random.randint(1000000, 8000000) # CIP Ficticio
            incidente['antecedentes_victimario'] = np.random.choice(prob_antecedentes_victimario.index, p=prob_antecedentes_victimario.values)
            incidente['reiterante'] = np.random.choice(prob_reiterante.index, p=prob_reiterante.values)
        else:
            incidente['edad_victimario'] = np.nan
            incidente['sexo_victimario'] = np.nan
            incidente['cip_victimario'] = np.nan
            incidente['antecedentes_victimario'] = np.nan
            incidente['reiterante'] = np.nan

        nuevos_incidentes.append(incidente)

df_nuevos = pd.DataFrame(nuevos_incidentes)
print(f"\nSe generaron un total de {len(df_nuevos)} nuevos registros sintéticos.")

# -----------------------------------------------------------------------------
# Paso 4: Combinación y Guardado del Dataset Ampliado y Completo
# -----------------------------------------------------------------------------
print("\nCombinando datos originales y sintéticos...")

# Definir el orden final y correcto de las columnas
columnas_finales = [
    'lat', 'lon', 'fecha', 'hora', 'minuto', 'dia_semana', 'tipo_incidente',
    'tipo_arma_fuego', 'tipo_local', 'edad_victima', 'sexo_victima',
    'cip_victima', 'antecedentes_victima', 'aprehension_detencion_victimario',
    'edad_victimario', 'sexo_victimario', 'cip_victimario',
    'antecedentes_victimario', 'reiterante'
]

# Asegurar que ambos dataframes tengan exactamente estas columnas en este orden
df_nuevos_final = df_nuevos[columnas_finales]
df_original_final = df_original[columnas_finales]

# Combinar los dataframes
df_ampliado = pd.concat([df_nuevos_final, df_original_final], ignore_index=True)
df_ampliado['fecha'] = pd.to_datetime(df_ampliado['fecha']).dt.date
df_ampliado.sort_values(by='fecha', inplace=True)

print("¡Combinación exitosa!")
print(f"  - Número total de registros: {len(df_ampliado)}")
print(f"  - Fecha del primer registro: {df_ampliado['fecha'].min()}")
print(f"  - Fecha del último registro: {df_ampliado['fecha'].max()}")
print(f"  - Columnas del dataset final: {df_ampliado.columns.tolist()}")

# Guardar el resultado en un nuevo archivo CSV
output_filename = 'incidentes_ampliados_7_anios_completo.csv'
df_ampliado.to_csv(output_filename, index=False, encoding='utf-8')

print(f"\nProceso completado. El dataset ampliado y completo ha sido guardado como '{output_filename}'.")

Librerías importadas correctamente.
Cargados 500 registros del archivo original.

Analizando el dataset original para aprender patrones y rangos...
Año de inicio de datos originales: 2024
Se generarán ~500 incidentes por cada nuevo año.
Análisis de patrones y distribuciones completado.

Generando datos sintéticos para los 6 años anteriores...
  - Generando datos para el año 2023...
  - Generando datos para el año 2022...
  - Generando datos para el año 2021...
  - Generando datos para el año 2020...
  - Generando datos para el año 2019...
  - Generando datos para el año 2018...

Se generaron un total de 3014 nuevos registros sintéticos.

Combinando datos originales y sintéticos...
¡Combinación exitosa!
  - Número total de registros: 3514
  - Fecha del primer registro: 2018-01-01
  - Fecha del último registro: 2024-12-31
  - Columnas del dataset final: ['lat', 'lon', 'fecha', 'hora', 'minuto', 'dia_semana', 'tipo_incidente', 'tipo_arma_fuego', 'tipo_local', 'edad_victima', 'sexo_victima